# Symptrack — Model Comparison (GRU variants vs ClinicalBERT)

This notebook builds several sequence models **from scratch** and compares them.
The goal is to justify our choice of the pre-trained ClinicalBERT model by showing
how much prior language knowledge (embeddings) matters on a small dataset.

All models use the **same data split** as ClinicalBERT for a fair comparison.

## 1. Load Data and Split
We load the mapped dataset, apply the **same stratified train/val/test split** used for ClinicalBERT,
and add the 60 short training examples to the training set only (to match ClinicalBERT and avoid leakage).

Final training distribution: ER = 158, Doctor = 518, Self-care = 191.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scikit-learn tensorflow

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

LABEL2ID = {"ER": 0, "Doctor": 1, "Self-care": 2}

# الداتا الأصلية
full_df = pd.read_csv("/content/drive/MyDrive/Symptrack/symptom2disease_triage_mapped.csv")
if "label_id" not in full_df.columns:
    full_df["label_id"] = full_df["triage_label"].map(LABEL2ID)

X, y = full_df["text"], full_df["label_id"]

# نفس التقسيم بالضبط زي ClinicalBERT
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

# نضيف الأمثلة القصيرة للـ train فقط (نفس ما سوينا مع ClinicalBERT)
short_df = pd.read_csv("/content/drive/MyDrive/Symptrack/short_text_augmentation.csv")
X_train = pd.concat([X_train, short_df["text"]], ignore_index=True)
y_train = pd.concat([y_train, short_df["label_id"]], ignore_index=True)

print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))
print("Train distribution:", y_train.value_counts().sort_index().to_dict())

Train: 867 | Val: 173 | Test: 173
Train distribution: {0: 158, 1: 518, 2: 191}


## 2. Tokenization
Unlike ClinicalBERT (which has its own tokenizer), a from-scratch model needs its own vocabulary.
We build a word index from the **training set only** and pad all sequences to length 80.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 5000
MAX_LEN = 80

tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(X_train)   # نتعلّم القاموس من الـ train فقط

def to_seq(texts):
    return pad_sequences(tok.texts_to_sequences(texts), maxlen=MAX_LEN, padding="post", truncating="post")

X_train_seq = to_seq(X_train)
X_val_seq   = to_seq(X_val)
X_test_seq  = to_seq(X_test)

y_train_arr = np.array(y_train)
y_val_arr   = np.array(y_val)
y_test_arr  = np.array(y_test)

print("Sequences ready:", X_train_seq.shape)

Sequences ready: (867, 80)


## 3. Model 1 — GRU without pre-trained embeddings
First attempt: a plain GRU that learns word meanings from scratch.

**Result: this failed.** With only 867 training samples, the model could not learn word meanings
and ended up predicting the majority class (Doctor) for almost everything.
We kept it here to document the starting point of our comparison.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    GRU(64, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

cw = compute_class_weight("balanced", classes=np.array([0,1,2]), y=y_train_arr)
class_weights = {0: cw[0], 1: cw[1], 2: cw[2]}
print("Class weights:", class_weights)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Class weights: {0: np.float64(1.8291139240506329), 1: np.float64(0.5579150579150579), 2: np.float64(1.513089005235602)}


We retried the same idea with a higher learning rate and different settings, but it still failed for the same reason (too little data, no prior word knowledge).

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model2 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128),
    GRU(64, dropout=0.2, recurrent_dropout=0.0),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

model2.compile(optimizer=Adam(learning_rate=1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

history = model2.fit(
    X_train_seq, y_train_arr,
    validation_data=(X_val_seq, y_val_arr),
    epochs=30, batch_size=16,
    class_weight=class_weights,
    callbacks=[early_stop], verbose=1
)

Epoch 1/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.4902 - loss: 1.1009 - val_accuracy: 0.1676 - val_loss: 1.1124
Epoch 2/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3022 - loss: 1.1019 - val_accuracy: 0.1676 - val_loss: 1.1087
Epoch 3/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2837 - loss: 1.1025 - val_accuracy: 0.1676 - val_loss: 1.0952
Epoch 4/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3195 - loss: 1.1014 - val_accuracy: 0.2139 - val_loss: 1.0948
Epoch 5/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3795 - loss: 1.1003 - val_accuracy: 0.1676 - val_loss: 1.1029
Epoch 6/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3379 - loss: 1.1014 - val_accuracy: 0.6185 - val_loss: 1.0963
Epoch 7/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2434 - loss: 1.1010 - val_accuracy: 0.1676 - val_loss: 1.1117
Epoch 8/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2884 - loss: 1.1007 - val_accuracy: 0.2139 - va

### Model 1 result on the test set
Macro-F1 = 0.255, ER recall = 0.000 — the model predicted Doctor for every case.

In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

gru_probs = model2.predict(X_test_seq)
gru_preds = np.argmax(gru_probs, axis=1)

print(classification_report(y_test_arr, gru_preds, target_names=["ER","Doctor","Self-care"], digits=3))
print("Macro-F1:", round(f1_score(y_test_arr, gru_preds, average="macro"), 3))

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step
              precision    recall  f1-score   support

          ER      0.000     0.000     0.000        29
      Doctor      0.618     1.000     0.764       107
   Self-care      0.000     0.000     0.000        37

    accuracy                          0.618       173
   macro avg      0.206     0.333     0.255       173
weighted avg      0.383     0.618     0.473       173

Macro-F1: 0.255


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 4. Model 2 — GRU + GloVe (general pre-trained embeddings)
To fix the failure above, we added **GloVe** embeddings, which give each word a ready meaning
learned from billions of general-domain words. This let the GRU focus on classification instead
of learning language from scratch.

**Result: big improvement — Macro-F1 jumped from 0.255 to 0.931.**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 1. نحمّل GloVe embeddings (معاني كلمات جاهزة)
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.100d.txt

embeddings_index = {}
with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        embeddings_index[values[0]] = np.asarray(values[1:], dtype="float32")
print("Loaded", len(embeddings_index), "word vectors")

# 2. نبني مصفوفة الـ embeddings لكلماتنا
EMBED_DIM = 100
word_index = tok.word_index
num_words = min(MAX_WORDS, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, EMBED_DIM))
for word, i in word_index.items():
    if i < num_words:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec

# 3. نبني الموديل مع GloVe + Bidirectional
model3 = Sequential([
    Embedding(num_words, EMBED_DIM, weights=[embedding_matrix],
              input_length=MAX_LEN, trainable=True),
    Bidirectional(GRU(64, dropout=0.3)),
    Dense(64, activation="relu"),
    Dropout(0.4),
    Dense(3, activation="softmax")
])

model3.compile(optimizer=Adam(learning_rate=5e-4),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True)

history = model3.fit(
    X_train_seq, y_train_arr,
    validation_data=(X_val_seq, y_val_arr),
    epochs=40, batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop], verbose=1
)
print("\n=== DONE ===")

Loaded 400000 word vectors
Epoch 1/40


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.4083 - loss: 1.0997 - val_accuracy: 0.4393 - val_loss: 1.0670
Epoch 2/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4095 - loss: 1.0681 - val_accuracy: 0.5491 - val_loss: 1.0037
Epoch 3/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4694 - loss: 1.0356 - val_accuracy: 0.5491 - val_loss: 0.9793
Epoch 4/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5144 - loss: 1.0176 - val_accuracy: 0.5434 - val_loss: 0.9904
Epoch 5/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5075 - loss: 0.9891 - val_accuracy: 0.6185 - val_loss: 0.9162
Epoch 6/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5340 - loss: 0.9582 - val_accuracy: 0.6185 - val_loss: 0.9064
Epoch 7/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5386 - loss: 0.9239 - val_accuracy: 0.6532 - val_loss: 0.8531
Epoch 8/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5744 - loss: 0.8656 - val_accuracy: 0.7168 - val_loss: 0.

### Model 2 result on the test set
Macro-F1 = 0.931, ER recall = 0.897.

In [ ]:
from sklearn.metrics import f1_score, recall_score, classification_report
import numpy as np

gru_probs = model3.predict(X_test_seq)
gru_preds = np.argmax(gru_probs, axis=1)

print(classification_report(y_test_arr, gru_preds, target_names=["ER","Doctor","Self-care"], digits=3))
print("Macro-F1:", round(f1_score(y_test_arr, gru_preds, average="macro"), 3))
print("ER Recall:", round(recall_score(y_test_arr, gru_preds, labels=[0], average="micro"), 3))

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step
              precision    recall  f1-score   support

          ER      0.839     0.897     0.867        29
      Doctor      0.971     0.935     0.952       107
   Self-care      0.949     1.000     0.974        37

    accuracy                          0.942       173
   macro avg      0.919     0.944     0.931       173
weighted avg      0.944     0.942     0.943       173

Macro-F1: 0.931
ER Recall: 0.897


### Testing GRU+GloVe on real short inputs
4 of 5 cases were correct. The only miss was the ambiguous 'mild cough' case, which it sent to ER —
because static embeddings give 'cough' a fixed emergency-related meaning regardless of context.

In [ ]:
import numpy as np

id2label = {0: "ER", 1: "Doctor", 2: "Self-care"}

def predict_gru(text):
    seq = to_seq([text])   # نفس دالة التحويل
    probs = model3.predict(seq, verbose=0)[0]
    pred = np.argmax(probs)
    print("Text:", text)
    print("ER: {:.1%} | Doctor: {:.1%} | Self-care: {:.1%}".format(probs[0], probs[1], probs[2]))
    print("Prediction:", id2label[pred])
    print("-"*50)

# نفس الحالات اللي جربناها على ClinicalBERT
predict_gru("I have a mild runny nose and slight cough since yesterday")
predict_gru("I have mild acne on my face")
predict_gru("I have a small itchy skin rash on my arm")
predict_gru("I have severe chest pain and difficulty breathing")
predict_gru("I've had joint pain and stiffness for two weeks")

Text: I have a mild runny nose and slight cough since yesterday
ER: 88.4% | Doctor: 0.2% | Self-care: 11.4%
Prediction: ER
--------------------------------------------------
Text: I have mild acne on my face
ER: 4.7% | Doctor: 0.9% | Self-care: 94.4%
Prediction: Self-care
--------------------------------------------------
Text: I have a small itchy skin rash on my arm
ER: 1.3% | Doctor: 0.6% | Self-care: 98.1%
Prediction: Self-care
--------------------------------------------------
Text: I have severe chest pain and difficulty breathing
ER: 99.9% | Doctor: 0.1% | Self-care: 0.0%
Prediction: ER
--------------------------------------------------
Text: I've had joint pain and stiffness for two weeks
ER: 4.7% | Doctor: 94.3% | Self-care: 1.0%
Prediction: Doctor
--------------------------------------------------


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import numpy as np

def per_class_metrics(y_true, y_pred, model_name):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0,1,2])
    return pd.DataFrame({
        "Model": model_name,
        "Class": ["ER", "Doctor", "Self-care"],
        "Precision": np.round(p, 3),
        "Recall": np.round(r, 3),
        "F1": np.round(f, 3),
    })

# ClinicalBERT — تحتاجين test_preds من موديل BERT
# (لو مو موجود في هذا النوتبوك، شغّليه في نوتبوك BERT)

# GRU
gru_table = per_class_metrics(y_test_arr, gru_preds, "GRU+GloVe")
print(gru_table.to_string(index=False))

    Model     Class  Precision  Recall    F1
GRU+GloVe        ER      0.839   0.897 0.867
GRU+GloVe    Doctor      0.971   0.935 0.952
GRU+GloVe Self-care      0.949   1.000 0.974


## 5. Save Models
We save the from-scratch GRU (for documentation) and the GRU+GloVe model (with its tokenizer,
which is required to use the model later).

In [ ]:
import os, pickle
BASE = "/content/drive/MyDrive/Symptrack/"

# 1. GRU بدون GloVe (للتوثيق — النسخة الفاشلة)
d1 = BASE + "gru_basic_model"
os.makedirs(d1, exist_ok=True)
model2.save(d1 + "/gru_basic.keras")
with open(d1 + "/tokenizer.pickle", "wb") as f:
    pickle.dump(tok, f)
print("Saved GRU (basic, no embeddings) to:", d1)

# 2. GRU + GloVe (النسخة الشغّالة)
d2 = BASE + "gru_glove_model"
os.makedirs(d2, exist_ok=True)
model3.save(d2 + "/gru_glove.keras")
with open(d2 + "/tokenizer.pickle", "wb") as f:
    pickle.dump(tok, f)
print("Saved GRU+GloVe to:", d2)

Saved GRU (basic, no embeddings) to: /content/drive/MyDrive/Symptrack/gru_basic_model
Saved GRU+GloVe to: /content/drive/MyDrive/Symptrack/gru_glove_model


## 6. Model 3 — GRU + BioWordVec (medical pre-trained embeddings)
We then tried **BioWordVec**, embeddings trained on medical text (PubMed + MIMIC-III),
to see whether medical-domain word vectors beat general ones.

1358 of 1463 of our words were found in BioWordVec.

In [ ]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 74.9 MB/s eta 0:00:00


In [ ]:
# BioWordVec — نسخة الـ embeddings الطبية
import gensim
import numpy as np

# نحمّل نسخة أخف (intrinsic) — أصغر من الكاملة
!wget -q https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioWordVec_PubMed_MIMICIII_d200.vec.bin

print("Downloaded. Loading...")
bio_model = gensim.models.KeyedVectors.load_word2vec_format(
    "BioWordVec_PubMed_MIMICIII_d200.vec.bin", binary=True, limit=200000
)
print("Loaded BioWordVec:", len(bio_model.index_to_key), "words")

Downloaded. Loading...
Loaded BioWordVec: 200000 words


In [ ]:
import numpy as np

EMBED_DIM_BIO = 200
word_index = tok.word_index
num_words = min(MAX_WORDS, len(word_index) + 1)

embedding_matrix_bio = np.zeros((num_words, EMBED_DIM_BIO))
found = 0
for word, i in word_index.items():
    if i < num_words:
        if word in bio_model:
            embedding_matrix_bio[i] = bio_model[word]
            found += 1
print(f"Found {found} of {num_words} words in BioWordVec")

Found 1358 of 1463 words in BioWordVec


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model4 = Sequential([
    Embedding(num_words, EMBED_DIM_BIO, weights=[embedding_matrix_bio],
              input_length=MAX_LEN, trainable=True),
    Bidirectional(GRU(64, dropout=0.3)),
    Dense(64, activation="relu"),
    Dropout(0.4),
    Dense(3, activation="softmax")
])
model4.compile(optimizer=Adam(learning_rate=5e-4),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True)
history4 = model4.fit(
    X_train_seq, y_train_arr,
    validation_data=(X_val_seq, y_val_arr),
    epochs=40, batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop], verbose=1
)
print("\n=== BioWordVec DONE ===")

Epoch 1/40


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.3806 - loss: 1.0858 - val_accuracy: 0.5145 - val_loss: 1.0460
Epoch 2/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4764 - loss: 1.0581 - val_accuracy: 0.4971 - val_loss: 1.0234
Epoch 3/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4983 - loss: 1.0359 - val_accuracy: 0.5318 - val_loss: 1.0059
Epoch 4/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4902 - loss: 0.9930 - val_accuracy: 0.6243 - val_loss: 0.8968
Epoch 5/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5675 - loss: 0.9233 - val_accuracy: 0.6763 - val_loss: 0.8243
Epoch 6/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6378 - loss: 0.8351 - val_accuracy: 0.7514 - val_loss: 0.6728
Epoch 7/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6667 - loss: 0.7198 - val_accuracy: 0.7630 - val_loss: 0.5993
Epoch 8/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7243 - loss: 0.6414 - val_accuracy: 0.7688 - val_loss: 0.

### Model 3 result on the test set
Macro-F1 = 0.926, ER recall = 0.931. Very close to GloVe overall, with slightly better ER recall.
But it still misclassified 'mild cough' as ER — the same context problem as GloVe.

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_recall_fscore_support
import numpy as np
import pandas as pd

bio_probs = model4.predict(X_test_seq, verbose=0)
bio_preds = np.argmax(bio_probs, axis=1)

print("=== GRU + BioWordVec ===")
from sklearn.metrics import classification_report
print(classification_report(y_test_arr, bio_preds, target_names=["ER","Doctor","Self-care"], digits=3))
print("Macro-F1:", round(f1_score(y_test_arr, bio_preds, average="macro"), 3))
print("ER Recall:", round(recall_score(y_test_arr, bio_preds, labels=[0], average="micro"), 3))

=== GRU + BioWordVec ===
              precision    recall  f1-score   support

          ER      0.794     0.931     0.857        29
      Doctor      0.981     0.944     0.962       107
   Self-care      0.972     0.946     0.959        37

    accuracy                          0.942       173
   macro avg      0.916     0.940     0.926       173
weighted avg      0.948     0.942     0.944       173

Macro-F1: 0.926
ER Recall: 0.931


In [ ]:
id2label = {0: "ER", 1: "Doctor", 2: "Self-care"}
def predict_bio(text):
    seq = to_seq([text])
    probs = model4.predict(seq, verbose=0)[0]
    print("Text:", text)
    print("ER: {:.1%} | Doctor: {:.1%} | Self-care: {:.1%}".format(probs[0], probs[1], probs[2]))
    print("→", id2label[np.argmax(probs)])
    print("-"*50)

predict_bio("I have a mild runny nose and slight cough since yesterday")
predict_bio("I have severe chest pain and difficulty breathing")
predict_bio("I have mild acne on my face")

Text: I have a mild runny nose and slight cough since yesterday
ER: 91.3% | Doctor: 1.4% | Self-care: 7.2%
→ ER
--------------------------------------------------
Text: I have severe chest pain and difficulty breathing
ER: 99.3% | Doctor: 0.6% | Self-care: 0.0%
→ ER
--------------------------------------------------
Text: I have mild acne on my face
ER: 14.2% | Doctor: 4.3% | Self-care: 81.5%
→ Self-care
--------------------------------------------------


## 7. Compare All Models on New Unseen Inputs
We test all GRU variants on 5 brand-new short cases. All variants over-predict ER on mild cases
(e.g. 'headache and tired'), confirming that static embeddings struggle with short, ambiguous context.

In [ ]:
import numpy as np
id2label = {0: "ER", 1: "Doctor", 2: "Self-care"}

new_examples = [
    "I have a dry cough and mild fever for two days",
    "I'm coughing up blood and struggling to breathe",
    "I feel dizzy and my heart is racing",
    "I have a headache and feel tired",
    "I have a deep cut on my hand that won't stop bleeding",
]

def compare_all(text):
    seq = to_seq([text])
    # الموديلات الثلاثة اللي في هذا النوتبوك
    p_basic = model2.predict(seq, verbose=0)[0]
    p_glove = model3.predict(seq, verbose=0)[0]
    p_bio   = model4.predict(seq, verbose=0)[0]

    print("TEXT:", text)
    print(f"  GRU (basic):     {id2label[np.argmax(p_basic)]:10s} (ER {p_basic[0]:.0%} | Doc {p_basic[1]:.0%} | Self {p_basic[2]:.0%})")
    print(f"  GRU + GloVe:     {id2label[np.argmax(p_glove)]:10s} (ER {p_glove[0]:.0%} | Doc {p_glove[1]:.0%} | Self {p_glove[2]:.0%})")
    print(f"  GRU + BioWordVec:{id2label[np.argmax(p_bio)]:10s} (ER {p_bio[0]:.0%} | Doc {p_bio[1]:.0%} | Self {p_bio[2]:.0%})")
    print("-"*70)

for ex in new_examples:
    compare_all(ex)

TEXT: I have a dry cough and mild fever for two days
  GRU (basic):     Doctor     (ER 33% | Doc 33% | Self 33%)
  GRU + GloVe:     ER         (ER 99% | Doc 0% | Self 0%)
  GRU + BioWordVec:ER         (ER 91% | Doc 9% | Self 0%)
----------------------------------------------------------------------
TEXT: I'm coughing up blood and struggling to breathe
  GRU (basic):     Doctor     (ER 33% | Doc 33% | Self 33%)
  GRU + GloVe:     ER         (ER 99% | Doc 0% | Self 0%)
  GRU + BioWordVec:ER         (ER 99% | Doc 0% | Self 1%)
----------------------------------------------------------------------
TEXT: I feel dizzy and my heart is racing
  GRU (basic):     Doctor     (ER 33% | Doc 33% | Self 33%)
  GRU + GloVe:     ER         (ER 99% | Doc 1% | Self 0%)
  GRU + BioWordVec:ER         (ER 99% | Doc 1% | Self 0%)
----------------------------------------------------------------------
TEXT: I have a headache and feel tired
  GRU (basic):     Doctor     (ER 33% | Doc 33% | Self 33%)
  GRU + Glo

## 8. Extra Experiment — Trying to Reduce the ER Bias (did not help)
We tried retraining GRU+GloVe with **lighter class weights** to reduce the tendency to over-predict ER.

**We did this, then found it did not fix the problem:** the ER bias stayed (mild cough and headache
were still classified as ER) and Macro-F1 slightly dropped to 0.911. This showed the bias comes from
the static embeddings not understanding context — not from the class weights. So we did not use this
version; it is kept only to document what we tried.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# أوزان أخف (بدل 1.83/0.56/1.51)
soft_weights = {0: 1.4, 1: 0.85, 2: 1.2}

model5 = Sequential([
    Embedding(num_words, 100, weights=[embedding_matrix], input_length=MAX_LEN, trainable=True),
    Bidirectional(GRU(64, dropout=0.3)),
    Dense(64, activation="relu"),
    Dropout(0.4),
    Dense(3, activation="softmax")
])
model5.compile(optimizer=Adam(learning_rate=5e-4),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True)
history5 = model5.fit(
    X_train_seq, y_train_arr,
    validation_data=(X_val_seq, y_val_arr),
    epochs=40, batch_size=32,
    class_weight=soft_weights,
    callbacks=[early_stop], verbose=1
)
print("\n=== IMPROVED GRU DONE ===")

Epoch 1/40


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.5663 - loss: 1.0510 - val_accuracy: 0.6185 - val_loss: 0.8951
Epoch 2/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.5894 - loss: 1.0515 - val_accuracy: 0.6590 - val_loss: 0.8945
Epoch 3/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5882 - loss: 1.0294 - val_accuracy: 0.6532 - val_loss: 0.8716
Epoch 4/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6113 - loss: 0.9915 - val_accuracy: 0.6936 - val_loss: 0.8519
Epoch 5/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6182 - loss: 0.9786 - val_accuracy: 0.6821 - val_loss: 0.8167
Epoch 6/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6367 - loss: 0.9485 - val_accuracy: 0.6763 - val_loss: 0.7983
Epoch 7/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6424 - loss: 0.8970 - val_accuracy: 0.7052 - val_loss: 0.7332
Epoch 8/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6724 - loss: 0.8529 - val_accuracy: 0.7110 - val_loss: 0.

In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

m5_preds = np.argmax(model5.predict(X_test_seq, verbose=0), axis=1)
print(classification_report(y_test_arr, m5_preds, target_names=["ER","Doctor","Self-care"], digits=3))
print("Macro-F1:", round(f1_score(y_test_arr, m5_preds, average="macro"), 3))

id2label = {0:"ER", 1:"Doctor", 2:"Self-care"}
def test5(text):
    probs = model5.predict(to_seq([text]), verbose=0)[0]
    print(f"{id2label[np.argmax(probs)]:10s} | ER {probs[0]:.0%} Doc {probs[1]:.0%} Self {probs[2]:.0%} | {text}")

print("\n--- Test cases ---")
test5("I have a dry cough and mild fever for two days")
test5("I have a headache and feel tired")
test5("I have mild acne on my face")
test5("I'm coughing up blood and struggling to breathe")
test5("I have a deep cut on my hand that won't stop bleeding")

              precision    recall  f1-score   support

          ER      0.730     0.931     0.818        29
      Doctor      0.980     0.935     0.957       107
   Self-care      1.000     0.919     0.958        37

    accuracy                          0.931       173
   macro avg      0.903     0.928     0.911       173
weighted avg      0.943     0.931     0.934       173

Macro-F1: 0.911

--- Test cases ---
ER         | ER 92% Doc 8% Self 0% | I have a dry cough and mild fever for two days
ER         | ER 92% Doc 7% Self 1% | I have a headache and feel tired
Self-care  | ER 41% Doc 12% Self 47% | I have mild acne on my face
ER         | ER 100% Doc 0% Self 0% | I'm coughing up blood and struggling to breathe
ER         | ER 99% Doc 1% Self 0% | I have a deep cut on my hand that won't stop bleeding


## 9. Conclusion

| Model | Macro-F1 | ER Recall |
|---|---|---|
| GRU (no embeddings) | 0.255 | 0.000 |
| GRU + GloVe | 0.931 | 0.897 |
| GRU + BioWordVec | 0.926 | 0.931 |
| ClinicalBERT (main model) | 0.979 | 0.966 |

Performance improved as the model's prior knowledge became more useful: no embeddings failed,
general embeddings (GloVe) worked well, and medical embeddings (BioWordVec) were similar with
slightly better ER recall. However, both static-embedding models still mishandled the ambiguous
'mild cough' case, while only the contextual ClinicalBERT read it correctly. **Contextual, medically
pre-trained language understanding — not just medical vocabulary — was the key to the best performance.**